<a href="https://colab.research.google.com/github/zune2/colab-notebooks/blob/main/2026-10-07-EP006-GPT-2-voca-llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer

# 1. 실제 LLM 토크나이저 로드 (GPT-2 기준)
# 대량의 데이터로 미리 학습된(Pre-trained) 어휘집을 그대로 가져옵니다.
tokenizer = AutoTokenizer.from_pretrained("gpt2")
vocab_size = tokenizer.vocab_size  # GPT-2의 어휘집 크기는 대략 50,257개입니다.

# 2. 데이터 정의 및 '단어(토큰)' 단위 변환
text = "Hello world! This is a simple Hello world example for LLM training."

# 텍스트를 인코딩 (문장 -> 토큰 ID 리스트)
tokens = tokenizer.encode(text)
print(f"원문: {text}")
print(f"토큰 ID 변환 결과: {tokens}")
print(f"분절된 토큰 형태: {[tokenizer.decode([t]) for t in tokens]}\n")

# 입력(X)과 정답(Y) 구축 (Next Token Prediction)
X = torch.tensor(tokens[:-1]).unsqueeze(0) # 마지막 토큰 제외
Y = torch.tensor(tokens[1:]).unsqueeze(0)  # 첫 번째 토큰 제외

# 3. LLM 아키텍처 정의 (어휘집 크기가 커졌으므로 모델 용량도 소폭 확장)
class TokenLLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, batch_first=True)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        out = self.embedding(x)

        # 미래 토큰 차단용 Causal Mask
        sz = x.size(1)
        mask = torch.triu(torch.full((sz, sz), float('-inf')), diagonal=1).to(x.device)

        out, _ = self.attn(out, out, out, attn_mask=mask, need_weights=False)
        logits = self.fc(out)
        return logits

# 4. 모델 및 학습 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TokenLLM(vocab_size).to(device)
X, Y = X.to(device), Y.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

# 5. 학습 루프 (문장이 길어지고 어휘가 많아져 Epoch을 늘림)
print("--- 학습 시작 ---")
for epoch in range(200):
    optimizer.zero_grad()
    outputs = model(X)

    loss = criterion(outputs.view(-1, vocab_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch+1}/200, Loss: {loss.item():.4f}")

# 6. 텍스트 생성 테스트 (Inference)
print("\n--- 생성 테스트 ---")
model.eval()
input_seq = "Hello world!" # 모델이 학습한 시작 문구
generated_tokens = tokenizer.encode(input_seq)

with torch.no_grad():
    for _ in range(10): # 뒤이어 올 10개의 '토큰'을 생성
        input_tensor = torch.tensor(generated_tokens).unsqueeze(0).to(device)
        logits = model(input_tensor)

        # 마지막 토큰 위치에서 가장 확률이 높은 다음 토큰 선택
        next_token_idx = torch.argmax(logits[0, -1, :]).item()
        generated_tokens.append(next_token_idx)

# 최종 토큰 ID 리스트를 사람이 읽을 수 있는 텍스트로 복원(디코딩)
final_text = tokenizer.decode(generated_tokens)
print(f"입력문장: '{input_seq}'")
print(f"LLM 생성결과: '{final_text}'")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

원문: Hello world! This is a simple Hello world example for LLM training.
토큰 ID 변환 결과: [15496, 995, 0, 770, 318, 257, 2829, 18435, 995, 1672, 329, 27140, 44, 3047, 13]
분절된 토큰 형태: ['Hello', ' world', '!', ' This', ' is', ' a', ' simple', ' Hello', ' world', ' example', ' for', ' LL', 'M', ' training', '.']

--- 학습 시작 ---
Epoch 40/200, Loss: 0.0011
Epoch 80/200, Loss: 0.0001
Epoch 120/200, Loss: 0.0001
Epoch 160/200, Loss: 0.0001
Epoch 200/200, Loss: 0.0001

--- 생성 테스트 ---
입력문장: 'Hello world!'
LLM 생성결과: 'Hello world! This is a simple Hello world example for LLM'


방금 작성한 미니 모델은 실제 GPT-2의 토크나이저(어휘집 크기 약 5만 개)를 쓰지만, 내부 신경망 크기(d_model=64)가 매우 작기 때문에 최신 노트북 CPU 기준으로 1~2초면 학습이 끝납니다.

이 모델이 한 번 학습(1 Forward/Backward Pass)할 때 컴퓨터가 얼마나 많이 움직여야 하는지, 파라미터 수와 연산량(FLOPs)을 직접 계산해 보겠습니다.

------------------------------

## 1. 파라미터(무게추) 수 계산

모델의 용량은 파라미터 개수로 결정됩니다. 우리 코드(TokenLLM)의 파라미터는 대부분 최종 출력 레이어(nn.Linear)에서 발생합니다.

* Embedding Layer: $VocabSize(50,257) \times d_{model}(64) \approx \mathbf{3.2M}$ (약 320만 개)
* Multihead Attention: $4 \times d_{model}(64) \times d_{model}(64) \approx \mathbf{16K}$ (약 1.6만 개)
* Linear (FC) Layer: $d_{model}(64) \times VocabSize(50,257) \approx \mathbf{3.2M}$ (약 320만 개)
* 총 파라미터 수: 약 6.4M (640만 개)

💡 참고: 실제 GPT-3가 175B(1,750억 개)인 것에 비하면 2만 배 이상 작은 초소형 크기입니다.

------------------------------

## 2. 1회 학습당 연산량 (FLOPs) 계산

FLOPs(Floating Point Operations)는 컴퓨터가 소수점 연산을 몇 번 하는지 나타내는 단위입니다. 딥러닝에서 대략적인 연산량은 다음과 같은 공식으로 추정합니다.
$$\text{Forward 연산량} \approx 2 \times \text{파라미터 수} \times \text{토큰 길이(Sequence Length)}$$
$$\text{Backward(역전파) 연산량} \approx 2 \times \text{Forward 연산량}$$
$$\text{1회 학습(Epoch) 총 연산량} \approx 6 \times \text{파라미터 수} \times \text{토큰 길이}$$
우리 예제 문장("Hello world! This is a simple...")의 토큰 길이는 대략 15개입니다.

* Forward 연산: 2 × 6,400,000 × 15 = 192,000,000 FLOPs (1억 9,200만 번)
* Backward 연산: Forward의 2배 = 384,000,000 FLOPs (3억 8,400만 번)
* 1개 Epoch 총 연산량: $1.92\text{억} + 3.84\text{억} = \mathbf{576,000,000\text{ FLOPs}}$ (약 5.7억 번)

------------------------------

## 3. CPU에서 실행하면 얼마나 걸릴까?

최신 노트북 CPU(Intel i5/i7 또는 Apple Silicon M 시리즈)의 연산 속도는 대략 100 GFLOPS (초당 1,000억 번 연산) 이상입니다.

* 1개 Epoch 연산 시간: $5.7\text{억 번 연산} \div 1,000\text{억 번/초} \approx \mathbf{0.0057\text{초}}$
* 200개 Epoch 전체 학습 시간: $0.0057\text{초} \times 200 \approx \mathbf{1.14\text{초}}$

메모리 이동이나 기타 오버헤드를 감안해도 CPU에서 단 1~2초면 학습이 완료되는 이유가 바로 여기에 있습니다. 대형 그래픽카드(GPU)가 전혀 필요 없는 가벼운 연산량입니다.

------------------------------

## 4. 파이썬 코드로 실제 연산량 측정해보기

이론적 계산 말고, PyTorch 도구를 사용해 내 컴퓨터에서 돌아가는 실제 모델의 파라미터 수와 연산량을 정밀하게 측정해 볼 수도 있습니다.

```
# !pip install ptflops

# 연산량 측정 라이브러리 설치 필요

from ptflops import get_model_profile
model = TokenLLM(vocab_size=50257)
# 모델의 프로파일(연산량, 파라미터) 출력macs, params = get_model_profile(model, input_size=(1, 15), as_strings=True, print_profile=False)
print(f"컴퓨터가 연산한 양 (MACs): {macs}") # FLOPs는 보통 MACs의 2배입니다.
print(f"총 파라미터 개수: {params}")
```

연산량을 보니 CPU로도 충분히 연습해 볼 만하죠? 다음 단계로 넘어가기 위해 어떤 주제에 도전해 보고 싶으신가요?

* 데이터 크기를 조금 더 키워서 CPU 학습의 한계(예: 소설 한 페이지 분량 학습) 경험해보기
* 연산량을 늘리지 않으면서 문장의 완성도를 높이는 Top-k / Temperature 샘플링 코드 추가하기
